# 03. Networks

In the previous notebooks, we have focused on single neuron models. However, one of the main strengths of Dendra is its ability to efficiently simulate large networks of neurons with complex morphologies. In this notebook, we will explore how to set up and simulate networks of branched morphologies using Dendra.

## General overview: event-based networks

- Dendra networks are event-driven: a pre-synaptic event (e.g., a voltage or gate crossing a threshold, or a NetStim spike) triggers a message to its post-synaptic targets.
- Each connection is managed by a NetCon, which:
  - Detects pre-synaptic events (intrinsic thresholds, scheduled spikes, or directly implemented in the presynaptic {py:class}`~dendra.models.mechanisms.Mechanism`),
  - Applies weights (and optional schedules) to shape the event amplitude,
  - Delays delivery via per-connection delay buffers.
- Post-synaptic compartments receive these weighted, delayed events as inputs to {py:class}`~dendra.models.mechanisms.Synapse` mechanisms (e.g., {py:class}`~dendra.models.mod.exp2syn`) which implement {py:meth}`~dendra.models.mechanisms.Synapse.net_receive`, which update their internal state and contribute currents to the membrane equation.
- Events are sparse in time: only connections with pre-synaptic activity enqueue deliveries, improving efficiency over dense timestep coupling.
- You can drive networks via intrinsic spiking thresholds (e.g., voltage crossings), stochastic activity ({py:class}`~dendra.models.networks.NetStim`), or explicit schedules ({py:meth}`~dendra.models.networks.NetStim.schedule`, {py:meth}`~dendra.models.networks.NetCon.schedule`).

## Example: Ring Network of Branched Neurons

We will create a simple ring network of branched neurons, where each neuron is connected to its two nearest neighbors. We will use the same branched morphology defined in the previous notebook and set up synaptic connections between the neurons.

In [ ]:
import dendra as dn
from dendra.models.mod import hh

model = dn.Tree.from_asc(
    "example.asc", 
    v_init=-65.0, 
    celsius=6.3,
    rhoa=100.0,
    N=5
)  # Load morphology from file, set initial voltage and temperature, create 5 instances

model.insert(hh)  # Insert Hodgkin-Huxley dynamics into all compartments


Now lets insert synapses and connect the neurons in a ring topology.

In [ ]:
from dendra.models.mod import exp2syn

model.soma.insert(exp2syn, tau1=0.2, tau2=1.0)  # Insert an exponential synapse at the soma of each neuron

net = dn.Network({"hh": model})  
# Create a network with the branched morphology model. The population is named "hh".

Now we connect in a ring topology.

In [ ]:
import torch
net.connect_one_to_one(
    net.hh.soma,
    net.hh.soma[torch.roll(torch.arange(5), shifts=-1)],
    net.hh.mech.exp2syn,
    threshold=0.0,
    weight=0.1,
    delay=5.0,
)

```{important}
We have surreptitiously introduced a powerful feature of Dendra here that we have not explicitly highlighted before: the ability to index into {py:class}`~dendra.models.slice.Slice`'s using PyTorch-style indexing. This allows for very flexible and efficient specification of connectivity patterns between neurons in a network. Here, we used `torch.roll` to create a ring topology by connecting each neuron's soma to its neighbor's soma in a circular fashion.
```

```{note}
We use a double exponential synapse model ({py:class}`~dendra.models.mod.exp2syn`) for the synaptic connections, with rising and falling time constants $\tau_1$ and $\tau_2$, respectively, that we have set explicitly (the defaults are 0.1 ms and 10.0 ms). You can explore other synaptic models available in Dendra or define your own custom synaptic models.
```

```{note}
We set the synaptic weight to `0.1` and the delay to `5.0 ms`. You can adjust these parameters to see how they affect the network dynamics.
```

We can now stimulate and run:

In [ ]:
from dendra.units import nA, ms

net.hh.delete_injections()
stim = dn.mono_rect(amp=2.0 * nA, pw=0.5 * ms, delay=1.0 * ms)
net.hh[0].soma.inject(stim)
# equivalently: net.hh.soma[0].inject(...)
rec = dn.callbacks.RecorderLambda(
    {"somas": lambda net: net.hh.soma.v}
)

tstop, dt = 40.0 * ms, 0.025 * ms

rec.reset()
net.initialize(dt=dt)
net.run(tstop=tstop, callbacks=[rec])

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6), dpi=300)

t = torch.arange(0, tstop+dt, dt)

soma_voltages = rec.numpy('somas')
for i in range(5):
    plt.plot(t, soma_voltages[:, i], linewidth=3, label=f'Neuron {i}')
plt.xlabel('Time (ms)')
plt.ylabel('Voltage (mV)')
plt.title('Soma Voltages of Neurons')
plt.legend()
plt.show()

Awesome! We have successfully set up and simulated a ring network of branched neurons using Dendra. We can see that with a single initial pulse delivered to the first neuron, the activity propagates through the network in a ring topology, eventually returning to the first neuron and continuing the cycle. We encourage you to further explore different network topologies (perhaps try connecting axonal compartments to dendritic compartments?), synaptic models, and stimulation protocols to study various aspects of neural network dynamics.